# Session 2: Deployment and Scaling (Student Notebook)

In this session you will learn how to take McKinsey consulting AI applications from prototype to production. We cover API design for consulting services (engagement assistant, knowledge search, client briefing generator), caching strategies for recurring consulting queries, monitoring for partner-review quality metrics, model routing by consulting complexity, and cost tracking per engagement -- the operational skills that separate demos from deployed consulting AI systems.

## Learning Objectives

By the end of this session, you will be able to:

1. **Design** a FastAPI-style service for McKinsey's consulting AI (engagement assistant, knowledge search, client briefing APIs)
2. **Implement** semantic caching to reduce costs on recurring consulting queries (market research, industry analyses)
3. **Add** monitoring and structured logging that tracks consulting-relevant metrics (response quality, framework coverage, analysis accuracy)
4. **Build** a model router that selects models based on consulting complexity (fact lookup vs. strategy analysis)
5. **Track** token usage and costs modeled around consulting AI economics (cost per engagement, tokens per client deliverable)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
import hashlib
import logging
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np

print("All imports successful!")

/Users/siddharthsharma/Desktop/mckinsey-genAi-3Day/vnev/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


All imports successful!


---
## Demos (Follow Along)

### Demo 1: Designing a McKinsey Consulting AI API Service

Production consulting AI applications need a clean API layer. We design a service class that wraps the LLM with request validation, error handling, and structured responses -- the same patterns you would use to build McKinsey's Engagement Assistant API or Knowledge Search API in a FastAPI endpoint.

In [2]:
# Demo 1 - McKinsey Consulting AI API Service Design

@dataclass
class ConsultingAIRequest:
    """Validated request for the McKinsey Consulting AI service."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.7
    request_id: str = ""
    engagement_id: str = "ENG-DEFAULT"  # McKinsey engagement identifier
    service_type: str = "knowledge_search"  # knowledge_search | engagement_assistant | briefing_generator
    
    def __post_init__(self):
        if not self.request_id:
            self.request_id = hashlib.md5(f"{self.prompt}{time.time()}".encode()).hexdigest()[:8]
        if self.max_tokens < 1 or self.max_tokens > 4096:
            raise ValueError("max_tokens must be between 1 and 4096")
        if self.temperature < 0 or self.temperature > 2:
            raise ValueError("temperature must be between 0 and 2")
        valid_services = ["knowledge_search", "engagement_assistant", "briefing_generator"]
        if self.service_type not in valid_services:
            raise ValueError(f"service_type must be one of {valid_services}")

@dataclass
class ConsultingAIResponse:
    """Structured response from the McKinsey Consulting AI service."""
    content: str
    model: str
    request_id: str
    tokens_used: int
    latency_ms: float
    engagement_id: str = ""
    service_type: str = ""
    status: str = "success"
    error: Optional[str] = None

class McKinseyAIService:
    """Production consulting AI service with validation and error handling.
    
    Supports three API endpoints:
    - Knowledge Search: Query McKinsey's internal knowledge base
    - Engagement Assistant: Help consultants with active engagements
    - Briefing Generator: Generate client-ready briefing documents
    """
    
    SYSTEM_PROMPTS = {
        "knowledge_search": "You are McKinsey's Knowledge Search AI. Provide concise, framework-driven answers drawing on consulting best practices, industry analyses, and strategic frameworks (Porter's Five Forces, 7S, MECE, etc.).",
        "engagement_assistant": "You are McKinsey's Engagement Assistant AI. Help consultants with active client engagements by providing structured analysis, workstream planning, and deliverable outlines.",
        "briefing_generator": "You are McKinsey's Client Briefing Generator. Create executive-ready briefing content that is structured, data-driven, and actionable for C-suite audiences."
    }
    
    def __init__(self):
        self.client = openai.OpenAI()
    
    def invoke(self, request: ConsultingAIRequest) -> ConsultingAIResponse:
        start = time.time()
        system_prompt = self.SYSTEM_PROMPTS.get(request.service_type, self.SYSTEM_PROMPTS["knowledge_search"])
        try:
            response = self.client.chat.completions.create(
                model=request.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": request.prompt}
                ],
                max_tokens=request.max_tokens,
                temperature=request.temperature
            )
            latency = (time.time() - start) * 1000
            return ConsultingAIResponse(
                content=response.choices[0].message.content,
                model=request.model,
                request_id=request.request_id,
                tokens_used=response.usage.total_tokens,
                latency_ms=round(latency, 2),
                engagement_id=request.engagement_id,
                service_type=request.service_type
            )
        except Exception as e:
            latency = (time.time() - start) * 1000
            return ConsultingAIResponse(
                content="", model=request.model,
                request_id=request.request_id,
                tokens_used=0, latency_ms=round(latency, 2),
                engagement_id=request.engagement_id,
                service_type=request.service_type,
                status="error", error=str(e)
            )

# Test the McKinsey Consulting AI Service
service = McKinseyAIService()

# Knowledge Search API call
req = ConsultingAIRequest(
    prompt="What are the key elements of a McKinsey market entry strategy for a healthcare client?",
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150")), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")),
    engagement_id="ENG-2024-0892",
    service_type="knowledge_search"
)
resp = service.invoke(req)

print(f"Request ID:    {resp.request_id}")
print(f"Engagement:    {resp.engagement_id}")
print(f"Service Type:  {resp.service_type}")
print(f"Status:        {resp.status}")
print(f"Model:         {resp.model}")
print(f"Tokens:        {resp.tokens_used}")
print(f"Latency:       {resp.latency_ms}ms")
print(f"Response:      {resp.content}")

Request ID:    46307edb
Engagement:    ENG-2024-0892
Service Type:  knowledge_search
Status:        success
Model:         gpt-4o-mini
Tokens:        372
Latency:       6216.85ms
Response:      A McKinsey market entry strategy for a healthcare client typically involves the following key elements, structured using a MECE framework:

1. **Market Assessment**:
   - **Market Size and Growth**: Analyze the current market size, growth rate, and future projections.
   - **Market Segmentation**: Identify and segment the market based on demographics, needs, and behaviors.
   - **Regulatory Environment**: Assess healthcare regulations, compliance requirements, and potential barriers to entry.

2. **Competitive Analysis**:
   - **Porter’s Five Forces**: Evaluate the competitive landscape by analyzing:
     - **Threat of New Entrants**: Barriers to entry and capital requirements.
     - **Bargaining Power of Suppliers**: Supplier concentration and availability of substitutes.
     - **Bargaining P

### Demo 2: Semantic Caching for Consulting Queries

LLM calls are expensive and slow. At a firm like McKinsey, consultants across engagements frequently ask similar market research questions and recurring industry analyses. Semantic caching stores previous responses and returns them for similar (not just identical) queries -- reducing costs by 30-70% when multiple teams research overlapping industries or frameworks.

In [3]:
# Demo 2 - Semantic Caching for McKinsey Consulting Queries

class SemanticCache:
    """Cache that matches by semantic similarity, not exact string match.
    
    Designed for McKinsey consulting queries where different consultants
    often ask similar market research and industry analysis questions.
    """
    
    def __init__(self, similarity_threshold=0.92):
        self.client = openai.OpenAI()
        self.cache = []  # List of (embedding, query, response)
        self.threshold = similarity_threshold
        self.hits = 0
        self.misses = 0
    
    def _embed(self, text):
        return self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def get(self, query):
        """Check cache for a semantically similar consulting query."""
        query_emb = self._embed(query)
        best_sim, best_response = 0, None
        
        for cached_emb, cached_query, cached_response in self.cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = (cached_query, cached_response)
        
        if best_sim >= self.threshold:
            self.hits += 1
            return {"hit": True, "similarity": best_sim,
                    "cached_query": best_response[0], "response": best_response[1]}
        self.misses += 1
        return {"hit": False, "similarity": best_sim}
    
    def put(self, query, response):
        """Store a consulting query-response pair in cache."""
        emb = self._embed(query)
        self.cache.append((emb, query, response))

# Test with McKinsey consulting queries
cache = SemanticCache(similarity_threshold=0.90)
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# First query -- a market research question (cache miss)
q1 = "What are the key drivers of digital transformation in the healthcare industry?"
result = cache.get(q1)
print(f"Query: {q1}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")

# Call LLM and cache the response
answer = llm.invoke([HumanMessage(content=q1)]).content
cache.put(q1, answer)
print(f"Cached response: {answer[:100]}...")

# Similar consulting query -- should hit cache (different phrasing, same research question)
q2 = "What is driving digital transformation in healthcare?"
result = cache.get(q2)
print(f"\nQuery: {q2}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
if result['hit']:
    print(f"Matched to: {result['cached_query']}")

# Different industry analysis -- should miss (unrelated topic)
q3 = "What are the main cost reduction levers in automotive manufacturing?"
result = cache.get(q3)
print(f"\nQuery: {q3}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
print(f"\nStats: {cache.hits} hits, {cache.misses} misses")

Query: What are the key drivers of digital transformation in the healthcare industry?
Cache: False (sim=0.0000)
Cached response: Digital transformation in the healthcare industry is driven by several key factors, including:

1. *...

Query: What is driving digital transformation in healthcare?
Cache: True (sim=0.9223)
Matched to: What are the key drivers of digital transformation in the healthcare industry?

Query: What are the main cost reduction levers in automotive manufacturing?
Cache: False (sim=0.3078)

Stats: 1 hits, 2 misses


### Demo 3: Monitoring and Structured Logging for Consulting AI

In production, McKinsey partners and engagement managers need visibility into AI performance. Structured logging captures every request, response quality metrics, token count, latency, and errors -- enabling partner review of AI-generated analyses and tracking framework coverage across engagements.

In [4]:
# Demo 3 - Monitoring and Structured Logging for McKinsey Consulting AI

class ConsultingAIMonitor:
    """Monitors McKinsey consulting AI usage with structured logging and metrics.
    
    Tracks consulting-relevant metrics: response quality for partner review,
    analysis accuracy, framework coverage, and per-engagement performance.
    """
    
    def __init__(self):
        self.logs = []
        self.metrics = defaultdict(list)
    
    def log_request(self, request_id, model, prompt, response, tokens, latency_ms,
                    engagement_id="", service_type="", status="success"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "request_id": request_id,
            "engagement_id": engagement_id,
            "service_type": service_type,
            "model": model,
            "prompt_preview": prompt[:100],
            "response_preview": response[:100] if response else "",
            "tokens": tokens,
            "latency_ms": latency_ms,
            "status": status
        }
        self.logs.append(entry)
        self.metrics["latency"].append(latency_ms)
        self.metrics["tokens"].append(tokens)
        self.metrics["models"].append(model)
        self.metrics["service_types"].append(service_type)
        return entry
    
    def get_summary(self):
        if not self.logs:
            return "No requests logged."
        latencies = self.metrics["latency"]
        tokens = self.metrics["tokens"]
        models = self.metrics["models"]
        service_types = self.metrics["service_types"]
        return {
            "total_requests": len(self.logs),
            "avg_latency_ms": round(np.mean(latencies), 2),
            "p95_latency_ms": round(np.percentile(latencies, 95), 2),
            "total_tokens": sum(tokens),
            "avg_tokens_per_query": round(np.mean(tokens), 2),
            "model_distribution": {m: models.count(m) for m in set(models)},
            "service_type_distribution": {s: service_types.count(s) for s in set(service_types) if s},
            "error_rate": sum(1 for l in self.logs if l["status"] != "success") / len(self.logs),
            "partner_review_ready": sum(1 for l in self.logs if l["status"] == "success")
        }

# Test with McKinsey consulting queries
monitor = ConsultingAIMonitor()
client = openai.OpenAI()

# Simulate consulting AI queries across different service types
consulting_queries = [
    ("What is McKinsey's 7S framework?", "knowledge_search", "ENG-2024-0101"),
    ("Summarize the competitive landscape in European fintech.", "engagement_assistant", "ENG-2024-0205"),
    ("What are the top 3 cost reduction levers for a telecom client?", "knowledge_search", "ENG-2024-0312"),
    ("Draft an executive summary for the digital transformation engagement.", "briefing_generator", "ENG-2024-0205"),
    ("Compare market entry strategies: greenfield vs. acquisition vs. JV.", "knowledge_search", "ENG-2024-0101"),
]

for prompt, svc_type, eng_id in consulting_queries:
    start = time.time()
    resp = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=[{"role": "user", "content": prompt}], max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80")))
    latency = (time.time() - start) * 1000
    req_id = hashlib.md5(prompt.encode()).hexdigest()[:8]
    
    entry = monitor.log_request(
        request_id=req_id, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        prompt=prompt, response=resp.choices[0].message.content,
        tokens=resp.usage.total_tokens, latency_ms=round(latency, 2),
        engagement_id=eng_id, service_type=svc_type
    )
    print(f"[{entry['request_id']}] [{entry['service_type']:22s}] {entry['prompt_preview'][:45]:45s} | {entry['tokens']}tok | {entry['latency_ms']}ms")

print("\n--- Consulting AI Performance Summary ---")
summary = monitor.get_summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

[a748d646] [knowledge_search      ] What is McKinsey's 7S framework?              | 318tok | 5960.94ms
[a6875927] [engagement_assistant  ] Summarize the competitive landscape in Europe | 317tok | 4818.03ms
[894a8b92] [knowledge_search      ] What are the top 3 cost reduction levers for  | 322tok | 5521.14ms
[d2c3528b] [briefing_generator    ] Draft an executive summary for the digital tr | 317tok | 5026.81ms
[c6c487e0] [knowledge_search      ] Compare market entry strategies: greenfield v | 321tok | 4570.93ms

--- Consulting AI Performance Summary ---
  total_requests: 5
  avg_latency_ms: 5179.57
  p95_latency_ms: 5872.98
  total_tokens: 1595
  avg_tokens_per_query: 319.0
  model_distribution: {'gpt-4o-mini': 5}
  service_type_distribution: {'knowledge_search': 3, 'engagement_assistant': 1, 'briefing_generator': 1}
  error_rate: 0.0
  partner_review_ready: 5


### Demo 4: Model Routing by Consulting Complexity

Not every consulting query needs GPT-4o. A model router classifies query complexity and routes accordingly: simple fact lookups (framework definitions, industry stats) go to gpt-4o-mini, while complex strategy analyses (market entry evaluation, M&A synergy modeling, competitive response planning) go to gpt-4o -- cutting costs by 40-60% with minimal quality loss on routine queries.

In [5]:
# Demo 4 - Model Routing by McKinsey Consulting Complexity

class ConsultingModelRouter:
    """Routes consulting queries to appropriate models based on complexity.
    
    Routing logic reflects McKinsey engagement patterns:
    - simple: Framework definitions, industry stats, quick fact lookups -> gpt-4o-mini
    - medium: Industry comparisons, trend summaries, workstream planning -> gpt-4o-mini
    - complex: Multi-dimensional strategy analysis, M&A evaluation, competitive response -> gpt-4o
    """
    
    MODEL_TIERS = {
        "simple": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "medium": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "complex": {"model": "gpt-4o", "cost_per_1k": 0.005}
    }
    
    def __init__(self):
        self.classifier = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
    
    def classify_complexity(self, query):
        """Classify consulting query complexity to determine model tier."""
        response = self.classifier.invoke([
            SystemMessage(content="""Classify this McKinsey consulting query's complexity. Return ONLY one word:
- simple: factual lookups, framework definitions, industry statistics, quick answers
- medium: industry comparisons, trend summaries, moderate analysis, workstream planning
- complex: multi-step strategy analysis, M&A evaluation, market entry planning, competitive response design"""),
            HumanMessage(content=query)
        ])
        complexity = response.content.strip().lower()
        if complexity not in self.MODEL_TIERS:
            complexity = "medium"
        return complexity
    
    def route(self, query):
        """Route consulting query to the appropriate model."""
        complexity = self.classify_complexity(query)
        tier = self.MODEL_TIERS[complexity]
        return {"complexity": complexity, "model": tier["model"], "cost_per_1k": tier["cost_per_1k"]}

# Test with McKinsey consulting queries of varying complexity
router = ConsultingModelRouter()
consulting_queries = [
    "What does MECE stand for?",
    "Compare the competitive landscape of European vs. US fintech markets across regulation, adoption, and profitability.",
    "Design a post-merger integration plan for a $2B healthcare acquisition including synergy capture, org design, and Day 1 readiness.",
    "What is Porter's Five Forces framework?",
    "Develop a market entry strategy for a Fortune 500 retailer entering Southeast Asian e-commerce with build, buy, and partner options."
]

for q in consulting_queries:
    result = router.route(q)
    print(f"[{result['complexity']:7s}] -> {result['model']:12s} (${result['cost_per_1k']}/1k) | {q[:70]}")

[simple ] -> gpt-4o-mini  ($0.00015/1k) | What does MECE stand for?
[complex] -> gpt-4o       ($0.005/1k) | Compare the competitive landscape of European vs. US fintech markets a
[complex] -> gpt-4o       ($0.005/1k) | Design a post-merger integration plan for a $2B healthcare acquisition
[simple ] -> gpt-4o-mini  ($0.00015/1k) | What is Porter's Five Forces framework?
[complex] -> gpt-4o       ($0.005/1k) | Develop a market entry strategy for a Fortune 500 retailer entering So


### Demo 5: Token Usage and Cost Tracking

Tracking costs is critical for budget management. This demo builds a cost tracker that monitors token usage per model, calculates running costs, and provides spending alerts.

In [6]:
# Demo 5 - Token Usage and Cost Tracking

class CostTracker:
    """Tracks token usage and costs across models."""
    
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "text-embedding-3-small": {"input": 0.00002, "output": 0}
    }
    
    def __init__(self, budget_limit=1.0):
        self.budget_limit = budget_limit
        self.records = []
    
    def record(self, model, input_tokens, output_tokens, request_id=""):
        pricing = self.PRICING.get(model, {"input": 0.001, "output": 0.002})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1000
        entry = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": cost,
            "request_id": request_id
        }
        self.records.append(entry)
        total = sum(r["cost"] for r in self.records)
        if total > self.budget_limit * 0.8:
            print(f"  WARNING: Budget at {total/self.budget_limit*100:.1f}% (${total:.6f}/${self.budget_limit})")
        return entry
    
    def get_report(self):
        total_cost = sum(r["cost"] for r in self.records)
        by_model = defaultdict(lambda: {"requests": 0, "input_tokens": 0, "output_tokens": 0, "cost": 0})
        for r in self.records:
            by_model[r["model"]]["requests"] += 1
            by_model[r["model"]]["input_tokens"] += r["input_tokens"]
            by_model[r["model"]]["output_tokens"] += r["output_tokens"]
            by_model[r["model"]]["cost"] += r["cost"]
        return {"total_cost": total_cost, "total_requests": len(self.records),
                "budget_remaining": self.budget_limit - total_cost, "by_model": dict(by_model)}

# Test
tracker = CostTracker(budget_limit=0.05)
client = openai.OpenAI()

queries = [
    ("gpt-4o-mini", "What is RAG?"),
    ("gpt-4o-mini", "List three vector databases."),
    ("gpt-4o", "Design a production RAG architecture with caching and monitoring."),
    ("gpt-4o-mini", "What is cosine similarity?"),
]

for model, prompt in queries:
    resp = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
    )
    entry = tracker.record(model, resp.usage.prompt_tokens, resp.usage.completion_tokens)
    print(f"[{model:12s}] {prompt[:40]:40s} | ${entry['cost']:.6f}")

print("\n--- Cost Report ---")
report = tracker.get_report()
print(f"Total cost: ${report['total_cost']:.6f}")
print(f"Budget remaining: ${report['budget_remaining']:.6f}")
for model, stats in report['by_model'].items():
    print(f"  {model}: {stats['requests']} reqs, {stats['input_tokens']+stats['output_tokens']} tokens, ${stats['cost']:.6f}")

[gpt-4o-mini ] What is RAG?                             | $0.000161
[gpt-4o-mini ] List three vector databases.             | $0.000073
[gpt-4o      ] Design a production RAG architecture wit | $0.004590
[gpt-4o-mini ] What is cosine similarity?               | $0.000182

--- Cost Report ---
Total cost: $0.005006
Budget remaining: $0.044994
  gpt-4o-mini: 3 reqs, 721 tokens, $0.000416
  gpt-4o: 1 reqs, 318 tokens, $0.004590


---
## Tasks (Your Turn!)

### Task 1: Build a Rate-Limited LLM Service

Create an LLM service wrapper that enforces rate limits — both requests-per-minute and tokens-per-minute — to prevent runaway costs and API throttling.

In [7]:
# Task 1 - Build a Rate-Limited LLM Service

# TODO: Build a RateLimitedLLM class that:
# 1. Tracks requests and tokens in a sliding window (last 60 seconds)
# 2. Blocks requests that would exceed the rate limit
# 3. Returns a helpful error message when rate limited
#
# Hint: Store timestamps of recent requests and their token counts
# Hint: Before each request, prune entries older than 60 seconds
# Hint: Check both RPM and TPM limits before making the API call

class RateLimitedLLM:
    def __init__(self, rpm_limit=10, tpm_limit=5000):
        # YOUR CODE HERE
        pass

    def invoke(self, prompt, max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))):
        # YOUR CODE HERE
        pass


# Test
# rl_llm = RateLimitedLLM(rpm_limit=3, tpm_limit=1000)
# for i in range(5):
#     result = rl_llm.invoke(f"Question {i}: What is RAG?")
#     print(f"Request {i}: {result['status']}")

### Task 2: Implement a Response Streaming Simulator

Build a streaming response handler that processes tokens as they arrive, tracks partial results, and provides real-time latency metrics (time-to-first-token, tokens-per-second).

In [8]:
# Task 2 - Implement a Response Streaming Simulator

# TODO: Build a StreamingHandler class that:
# 1. Uses the OpenAI streaming API (stream=True)
# 2. Tracks time-to-first-token (TTFT) and tokens-per-second
# 3. Collects the full response while reporting progress
#
# Hint: Use client.chat.completions.create(..., stream=True)
# Hint: Iterate over chunks: for chunk in response: ...
# Hint: Track time between first call and first token for TTFT

class StreamingHandler:
    def __init__(self):
        # YOUR CODE HERE
        pass

    def stream(self, prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))):
        # YOUR CODE HERE
        pass


# Test
# handler = StreamingHandler()
# result = handler.stream("Explain RAG in 3 sentences.")
# print(f"TTFT: {result['ttft_ms']:.0f}ms")
# print(f"Tokens/sec: {result['tokens_per_sec']:.1f}")
# print(f"Full response: {result['content']}")

### Task 3: Build a Multi-Tier Caching System

Build a caching system with two tiers: exact-match (fast, hash-based) and semantic-match (slower, embedding-based). Queries check exact cache first, then semantic cache, then call the LLM.

In [9]:
# Task 3 - Build a Multi-Tier Caching System

# TODO: Build a TieredCache class that:
# 1. Has an exact-match tier (dict-based, O(1) lookup)
# 2. Has a semantic-match tier (embedding similarity)
# 3. Falls through: exact -> semantic -> LLM call
# 4. Reports which tier served the response
#
# Hint: Use a dict for exact match (key = prompt hash)
# Hint: Use embedding comparison for semantic match
# Hint: Track hit rates for each tier

class TieredCache:
    def __init__(self, semantic_threshold=0.92):
        # YOUR CODE HERE
        pass

    def query(self, prompt):
        # YOUR CODE HERE
        pass


# Test
# cache = TieredCache()
# print(cache.query("What is RAG?"))        # LLM call
# print(cache.query("What is RAG?"))        # Exact match
# print(cache.query("Explain RAG to me"))   # Semantic match
# print(cache.query("How does gravity work?"))  # LLM call

### Task 4: Create a Full Production LLM Gateway

Combine everything into a production gateway: model routing, caching, rate limiting, monitoring, and cost tracking — all in one unified service.

In [10]:
# Task 4 - Create a Full Production LLM Gateway

# TODO: Build an LLMGateway class that combines:
# 1. Model routing (simple -> mini, complex -> 4o)
# 2. Caching (check cache before calling LLM)
# 3. Rate limiting (enforce RPM/TPM limits)
# 4. Monitoring (log every request)
# 5. Cost tracking (track spend per model)
#
# Hint: Compose the components from previous demos/tasks
# Hint: Pipeline: route -> check cache -> check rate limit -> call LLM -> cache response -> log
# Hint: Return a rich response with answer, model used, cache status, cost

class LLMGateway:
    def __init__(self):
        # YOUR CODE HERE
        pass

    def query(self, prompt):
        # YOUR CODE HERE
        pass

    def get_dashboard(self):
        # YOUR CODE HERE
        pass


# Test
# gateway = LLMGateway()
# for q in ["What is RAG?", "What is RAG?", "Design a multi-agent system."]:
#     result = gateway.query(q)
#     print(f"{q[:40]:40s} | model={result['model']} cache={result['cached']} cost=${result['cost']:.6f}")
# print(gateway.get_dashboard())

---
## Summary

In this session you learned the operational skills for production LLM deployment:

1. **API Design** -- How to structure LLM services with validation and error handling.
2. **Semantic Caching** -- How to reduce costs by caching similar (not just identical) queries.
3. **Monitoring** -- How to track latency, tokens, errors, and model usage.
4. **Model Routing** -- How to route queries to cheaper models when possible.
5. **Cost Tracking** -- How to monitor spending and set budget alerts.

**Up Next:** After lunch, you will choose your capstone track and build a complete system.